# Debugging Pandas

### Loading Libraries

In [ ]:
# Numerical Computing
import numpy as np

# Data Manipualtion
import pandas as pd

# Data Visualisation
import seaborn as sns
import matplotlib.pyplot as plt

# PyArrow
import pyarrow as pa

# URL
import urllib.request

# IPython
from IPython.display import display, HTML
from IPython.core.debugger import set_trace

In [ ]:
with open('/Users/joaquinromero/Desktop/EP2/data/devilclean.txt', 'r') as f:
    for i in range(10):
        print(repr(f.readline()))

In [ ]:
url = (
    'https://github.com/mattharrison/datasets/raw/master'
    '/data/dirtydevil.txt'
)

local_filename = '/Users/joaquinromero/Desktop/EP2/data/devilclean.txt'

urllib.request.urlretrieve(url, local_filename)

In [ ]:
with open(local_filename, 'r') as file:
    lines = file.readlines()

with open(local_filename, 'w') as file:
    for i, line in enumerate(lines):

        # elimina metadata del USGS
        if i < 34 or i == 35:
            continue

        file.write(line)

In [ ]:
with open(local_filename, 'r') as f:
    for i in range(5):
        print(repr(f.readline()))

In [ ]:
def to_denver_time(df_, time_col, tz_col):

    return (

        df_

        .assign(**{tz_col: df_[tz_col].replace('MDT', 'MST7MDT')})

        .groupby(tz_col)[time_col]

        .transform(

            lambda s:

                pd.to_datetime(s)

                .dt.tz_localize(s.name, ambiguous=True)

                .dt.tz_convert('America/Denver')

        )

    )

def tweak_river(df_):

    return (

        df_

        .assign(

            datetime=to_denver_time(df_, 'datetime', 'tz_cd')

        )

        .rename(

            columns={

                '144166_00060': 'cfs',

                '144167_00065': 'gage_height'

            }

        )

        .loc[

            :,

            [

                'datetime',

                'agency_cd',

                'site_no',

                'tz_cd',

                'cfs',

                'gage_height'

            ]

        ]

    )

#### Checking if DataFrames are Equal

In [ ]:
df = pd.read_csv('/Users/joaquinromero/Desktop/EP2/data/devilclean.txt', sep='\t', dtype_backend='pyarrow')

In [ ]:
dd = tweak_river(df)

print(dd)

In [ ]:
dd2 = pd.read_json(
    dd.to_json(),
    dtype_backend='pyarrow'
)

dd.equals(dd2)

In [ ]:
print(dd.eq(dd2))

In [ ]:
(dd
    .eq(dd2)
    .sum()
)

In [ ]:
pd.testing.assert_frame_equal(dd, dd2)

In [ ]:
from_json = (dd2
    .assign(datetime=dd2.datetime
        .dt.tz_localize('UTC')
        .dt.tz_convert('America/Denver'))
            )

In [ ]:
pd.testing.assert_frame_equal(dd, from_json)

In [ ]:
pd.testing.assert_frame_equal(
    dd,
    from_json,
    check_dtype=False
)

In [ ]:
dd.equals(from_json)

In [ ]:
pd.testing.assert_frame_equal(
    dd,
    from_json.astype(dict(dd.dtypes))
)

In [ ]:
dd.equals(from_json.astype(dict(dd.dtypes)))

In [ ]:
pd.testing.assert_frame_equal(
    dd,
    from_json.astype(dict(dd.dtypes)),
    check_exact=True
)

In [ ]:
dd3 = from_json.astype(dict(dd.dtypes))

In [ ]:
dd.eq(dd3).all()

In [ ]:
print(dd[dd.cfs.ne(dd3.cfs)])

In [ ]:
dd.loc[96246].cfs, dd3.loc[96246].cfs

In [ ]:
def cmp_dfs(df1, df2, round_amt=3):

    diff_cols = set(df1.columns) ^ set(df2.columns)

    if diff_cols:
        print(f'Different columns {diff_cols}')

    if df1.shape != df2.shape:
        print(f'Different shapes {df1.shape} {df2.shape}')

    bad = False

    for col in df1.columns:

        s1 = df1[col]
        s2 = df2[col]

        if s1.equals(s2):
            continue

        bad = True

        if s1.dtype != s2.dtype:

            print(
                f'{col} types differ {s1.dtype} vs {s2.dtype}'
            )

        if s1.dtype in ['float', 'double[pyarrow]']:

            if s1.round(round_amt).equals(
                s2.round(round_amt)
            ):

                print(
                    f'{col} has rounding differences'
                    f'{df1[s1.ne(s2)][col].dropna().iloc[0]} '
                    f'vs '
                    f'{df2[s1.ne(s2)][col].dropna().iloc[0]}'
                )

        else:

            diff = (
                df1
                .loc[s1.ne(s2)]
                .assign(other=s2)
                .loc[:, [col, 'other']]
                .dropna()
            )

            print(f'{col} differs {diff}')

    if not bad:
        print('Same')

In [ ]:
cmp_dfs(dd, dd3)

#### Debugging Chains

In [ ]:
import pandas as pd

autos = pd.read_csv(
    'https://github.com/mattharrison/datasets/raw/'
    'master/data/vehicles.csv.zip',
    dtype_backend='pyarrow',
    engine='pyarrow'
)

In [ ]:
def to_tz(df_, time_col, tz_offset, tz_name):
    return (df_
        .groupby(tz_offset)[time_col]
        .transform(
            lambda s: (
                pd.to_datetime(s)
                .dt.tz_localize(s.name, ambiguous=True)
                .dt.tz_convert(tz_name))
        )
    )

In [ ]:
def tweak_autos(autos):

    orig_cols = [
        'city08', 'comb08', 'highway08', 'cylinders',
        'displ', 'drive', 'eng_dscr', 'fuelCost08',
        'make', 'model', 'trany', 'range', 'createdOn',
        'year'
    ]

    types = {
        'highway08': 'int8[pyarrow]',
        'city08': 'int16[pyarrow]',
        'comb08': 'int16[pyarrow]',
        'fuelCost08': 'int16[pyarrow]',
        'range': 'int16[pyarrow]',
        'year': 'int16[pyarrow]',
        'make': 'category',
        'cylinders': 'int8[pyarrow]'
    }

    final_cols = [
        'city08', 'comb08', 'highway08', 'cylinders',
        'displ', 'drive', 'fuelCost08', 'make',
        'model', 'range', 'createdOn', 'year',
        'automatic', 'speeds', 'ffs'
    ]

    return (
        autos
        .loc[:, orig_cols]
        .assign(
            drive=autos.drive.fillna('Other').astype('category'),

            automatic=autos.trany.str.contains('Auto'),

            speeds=(
                autos.trany
                .str.extract(r'(?P<speeds>\d+)')
                .fillna('20')
                .astype('int8[pyarrow]')
            ),

            offset=(
                autos.createdOn
                .str.extract(r'\d:\d\d (?P<offset>[A-Z]{3})')
                .replace('EDT', 'EST5EDT')
            ),

            str_date=(
                autos.createdOn.str.slice(4, 19)
                + ' '
                + autos.createdOn.str.slice(-4)
            ),

            createdOn=lambda df_: to_tz(
                df_,
                'str_date',
                'offset',
                'America/New_York'
            ),

            ffs=autos.eng_dscr.str.contains('FFS')
        )
        .astype(types)
        .loc[:, final_cols]
    )

In [ ]:
def tweak_autos(autos):

    orig_cols = [
        'city08', 'comb08', 'highway08', 'cylinders',
        'displ', 'drive', 'eng_dscr', 'fuelCost08',
        'make', 'model', 'trany', 'range', 'createdOn',
        'year'
    ]

    return (
        autos
        .loc[:, orig_cols]
    )

In [ ]:
print(tweak_autos(autos))

In [ ]:
def tweak_autos(autos):

    orig_cols = [
        'city08', 'comb08', 'highway08', 'cylinders',
        'displ', 'drive', 'eng_dscr', 'fuelCost08',
        'make', 'model', 'trany', 'range', 'createdOn',
        'year'
    ]

    return (
        autos
        .loc[:, orig_cols]
        .assign(
            drive=autos.drive.fillna('Other').astype('category'),
        )
    )

In [ ]:
(
    tweak_autos(autos)
    .drive
    .value_counts()
)

In [ ]:
def tweak_autos(autos):

    orig_cols = [
        'city08', 'comb08', 'highway08', 'cylinders',
        'displ', 'drive', 'eng_dscr', 'fuelCost08',
        'make', 'model', 'trany', 'range', 'createdOn',
        'year'
    ]

    return (
        autos
        .loc[:, orig_cols]
        .assign(
            drive=autos.drive.fillna('Other').astype('category'),

            automatic=autos.trany.str.contains('Auto'),

            speeds=(
                autos.trany
                .str.extract(r'(?P<speeds>\d+)')
                .fillna('20')
                .astype('int8[pyarrow]')
            ),

            offset=(
                autos.createdOn
                .str.extract(r'\d:\d\d (?P<offset>[A-Z]{3})')
                .replace('EDT', 'EST5EDT')
            ),

            str_date=(
                autos.createdOn.str.slice(4, 19)
                + ' '
                + autos.createdOn.str.slice(-4)
            ),

            createdOn=lambda df_: to_tz(
                df_,
                'str_date',
                'offset',
                'America/New_York'
            ),

            ffs=autos.eng_dscr.str.contains('FFS')
        )
    )

In [ ]:
tweak_autos(autos).dtypes

In [ ]:
(tweak_autos(autos)
    .dtypes
    .value_counts()
)

In [ ]:
print(tweak_autos(autos)
    .describe()
    .T
)

In [ ]:
print(tweak_autos(autos)
    .select_dtypes('string')
    .nunique()
)

In [ ]:
def tweak_autos(autos):

    orig_cols = [
        'city08', 'comb08', 'highway08', 'cylinders',
        'displ', 'drive', 'eng_dscr', 'fuelCost08',
        'make', 'model', 'trany', 'range', 'createdOn',
        'year'
    ]

    types = {
        'highway08': 'int8[pyarrow]',
        'city08': 'int16[pyarrow]',
        'comb08': 'int16[pyarrow]',
        'fuelCost08': 'int16[pyarrow]',
        'range': 'int16[pyarrow]',
        'year': 'int16[pyarrow]',
        'make': 'category',
        'model': 'category',
        'cylinders': 'int8[pyarrow]'
    }

    return (
        autos
        .loc[:, orig_cols]
        .assign(
            drive=autos.drive.fillna('Other').astype('category'),

            automatic=autos.trany.str.contains('Auto'),

            speeds=(
                autos.trany
                .str.extract(r'(?P<speeds>\d+)')
                .fillna('20')
                .astype('int8[pyarrow]')
            ),

            offset=(
                autos.createdOn
                .str.extract(r'\d:\d\d (?P<offset>[A-Z]{3})')
                .replace('EDT', 'EST5EDT')
            ),

            str_date=(
                autos.createdOn.str.slice(4, 19)
                + ' '
                + autos.createdOn.str.slice(-4)
            ),

            createdOn=lambda df_: to_tz(
                df_,
                'str_date',
                'offset',
                'America/New_York'
            ),

            ffs=autos.eng_dscr.str.contains('FFS')
        )
        .astype(types)
    )

In [ ]:
tweak_autos(autos).dtypes

In [ ]:
def to_tz(df_, time_col, tz_offset, tz_name):

    return (
        df_
        .groupby(tz_offset)[time_col]
        .transform(
            lambda s: (
                pd.to_datetime(s)
                .dt.tz_localize(s.name, ambiguous=True)
                .dt.tz_convert(tz_name)
            )
        )
    )

In [ ]:
def tweak_autos(autos):

    orig_cols = [
        'city08', 'comb08', 'highway08', 'cylinders',
        'displ', 'drive', 'eng_dscr', 'fuelCost08',
        'make', 'model', 'trany', 'range', 'createdOn',
        'year'
    ]

    types = {
        'highway08': 'int8[pyarrow]',
        'city08': 'int16[pyarrow]',
        'comb08': 'int16[pyarrow]',
        'fuelCost08': 'int16[pyarrow]',
        'range': 'int16[pyarrow]',
        'year': 'int16[pyarrow]',
        'make': 'category',
        'cylinders': 'int8[pyarrow]'
    }

    final_cols = [
        'city08', 'comb08', 'highway08', 'cylinders',
        'displ', 'drive', 'fuelCost08', 'make', 'model',
        'range', 'createdOn', 'year', 'automatic',
        'speeds', 'ffs'
    ]

    return (
        autos
        [orig_cols]

        .assign(
            drive=autos.drive.fillna('Other').astype('category'),

            automatic=autos.trany.str.contains('Auto'),

            speeds=(
                autos.trany
                .str.extract(r'(?P<speeds>\d+)')
                .fillna('20')
                .astype('int8[pyarrow]')
            ),

            offset=(
                autos.createdOn
                .str.extract(r'\d\d:\d\d (?P<offset>[A-Z]{3}?)')
                .replace('EDT', 'EST5EDT')
            ),

            str_date=(
                autos.createdOn.str.slice(4, 19)
                + ' ' +
                autos.createdOn.str.slice(-4)
            ),

            createdOn=lambda df_: to_tz(
                df_,
                'str_date',
                'offset',
                'America/New_York'
            ),

            ffs=autos.eng_dscr.str.contains('FFS')
        )

        .astype(types)

        # 
        .drop(columns=['trany', 'eng_dscr', 'offset', 'str_date'])

        # 
        # .loc[:, final_cols]
    )

In [ ]:
tweak_autos(autos).columns

In [ ]:
print(tweak_autos(autos))

#### Debugging Chains Part II

In [ ]:
def show(df_, rows=20, cols=30, title=None):

    if title:
        display(HTML(f'<h2>{title}</h2>'))

    with pd.option_context(
        'display.min_rows', rows,
        'display.max_columns', cols
    ):
        display(df_)

    return df_

In [ ]:
# def tweak_autos(autos):

#     cols = [
#         'city08', 'comb08', 'highway08', 'cylinders',
#         'displ', 'drive', 'eng_dscr', 'fuelCost08',
#         'make', 'model', 'trany', 'range',
#         'createdOn', 'year'
#     ]

#     return (
#         autos
#         [cols]

#         .assign(
#             cylinders=autos.cylinders.fillna(0).astype('int8'),

#             displ=autos.displ.fillna(0).astype('float16'),

#             drive=autos.drive.fillna('Other').astype('category'),

#             automatic=autos.trany.str.contains('Auto'),

#             speeds=(
#                 autos.trany
#                 .str.extract(r'(\d)')
#                 .fillna('28')
#                 .astype('int8')
#             ),

#             tz=(
#                 autos.createdOn
#                 .str.extract(r'\d\d:\d\d ([A-Z]{3}?)')
#                 .replace('EDT', 'EST5EDT')
#             ),

#             str_date=(
#                 autos.createdOn.str.slice(4, 19)
#                 + ' ' +
#                 autos.createdOn.str.slice(-4)
#             ),

#             createdOn=lambda df_: to_tz(
#                 df_,
#                 'str_date',
#                 'tz',
#                 'US/Eastern'
#             ),

#             ffs=autos.eng_dscr.str.contains('FFS')
#         )

#         .pipe(show, rows=2, title='New Cols')

#         .astype({
#             'highway08': 'int8',
#             'city08': 'int16',
#             'comb08': 'int16',
#             'fuelCost08': 'int16',
#             'range': 'int16',
#             'year': 'int16',
#             'make': 'category'
#         })

#         .drop(columns=['trany', 'eng_dscr'])
#     )

In [ ]:
def to_tz(df_, time_col, tz_offset, tz_name):

    return (
        df_
        .groupby(tz_offset)[time_col]
        .transform(
            lambda s: (
                pd.to_datetime(s)
                .dt.tz_localize(s.name, ambiguous=True)
                .dt.tz_convert(tz_name)
            )
        )
    )


def tweak_autos(autos):

    cols = [
        'city08',
        'comb08',
        'highway08',
        'cylinders',
        'displ',
        'drive',
        'eng_dscr',
        'fuelCost08',
        'make',
        'model',
        'trany',
        'range',
        'createdOn',
        'year'
    ]

    return (
        autos
        [cols]

        .assign(

            cylinders=(
                autos.cylinders
                .fillna(0)
                .astype('int8[pyarrow]')
            ),

            displ=(
                autos.displ
                .fillna(0)
                .astype('float32[pyarrow]')
            ),

            drive=(
                autos.drive
                .fillna('Other')
                .astype('category')
            ),

            automatic=(
                autos.trany
                .str.contains('Auto')
            ),

            speeds=(
                autos.trany
                .str.extract(r'(?P<speeds>\d)')
                .fillna('28')
                .astype('int8[pyarrow]')
            ),

            tz=(
                autos.createdOn
                .str.extract(
                    r'\d\d:\d\d (?P<tz>[A-Z]{3}?)'
                )
                .replace('EDT', 'EST5EDT')
            ),

            str_date=(
                autos.createdOn.str.slice(4, 19)
                + ' ' +
                autos.createdOn.str.slice(-4)
            ),

            createdOn=lambda df_: to_tz(
                df_,
                'str_date',
                'tz',
                'US/Eastern'
            ),

            ffs=(
                autos.eng_dscr
                .str.contains('FFS')
            )
        )

        .pipe(show, rows=2, title='New Cols')

        .astype({
            'highway08': 'int8[pyarrow]',
            'city08': 'int16[pyarrow]',
            'comb08': 'int16[pyarrow]',
            'fuelCost08': 'int16[pyarrow]',
            'range': 'int16[pyarrow]',
            'year': 'int16[pyarrow]',
            'make': 'category'
        })

        .drop(columns=['trany', 'eng_dscr'])
    )

In [ ]:
tweak_autos(autos)

#### Debuggin Chains Part III

In [ ]:
cols = [
    'city08', 'comb08', 'highway08', 'cylinders',
    'displ', 'drive', 'eng_dscr', 'fuelCost08',
    'make', 'model', 'trany', 'range',
    'createdOn', 'year'
]

autos2 = autos[cols]

cyl_nona = autos.cylinders.fillna(0)

cyl_int8 = cyl_nona.astype('int8')

autos2['cylinders'] = cyl_int8

displ_nona = autos.displ.fillna(0)

displ_float16 = displ_nona.astype('float16')

autos2['displ'] = displ_float16

...

autos2.drop(
    columns=['trany', 'eng_dscr'],
    inplace=True
)

In [ ]:
def get_var(df, var_name):

    globals()[var_name] = df

    return df

In [ ]:
def to_tz(df_, time_col, tz_offset, tz_name):

    return (
        df_
        .groupby(tz_offset)[time_col]
        .transform(
            lambda s: (
                pd.to_datetime(s)
                .dt.tz_localize(s.name, ambiguous=True)
                .dt.tz_convert(tz_name)
            )
        )
    )

In [ ]:
def tweak_autos(autos):

    orig_cols = [
        'city08', 'comb08', 'highway08', 'cylinders',
        'displ', 'drive', 'eng_dscr', 'fuelCost08',
        'make', 'model', 'trany', 'range',
        'createdOn', 'year'
    ]

    types = {
        'highway08': 'int8[pyarrow]',
        'city08': 'int16[pyarrow]',
        'comb08': 'int16[pyarrow]',
        'fuelCost08': 'int16[pyarrow]',
        'range': 'int16[pyarrow]',
        'year': 'int16[pyarrow]',
        'make': 'category',
        'cylinders': 'int8[pyarrow]'
    }

    final_cols = [
        'city08',
        'comb08',
        'highway08',
        'cylinders',
        'displ',
        'drive',
        'fuelCost08',
        'make',
        'model',
        'range',
        'createdOn',
        'year',
        'automatic',
        'speeds',
        'ffs'
    ]

    return (
        autos
        [orig_cols]

        .assign(

            drive=(
                autos.drive
                .replace('', 'Other')
                .astype('category')
            ),

            automatic=(
                autos.trany
                .str.contains('Auto')
            ),

            speeds=(
                autos.trany
                .str.extract(r'(?P<speeds>\d+)')
                .fillna('20')
                .astype('int8[pyarrow]')
            ),

            offset=(
                autos.createdOn
                .str.extract(
                    r'\d\d:\d\d (?P<offset>[A-Z]{3}?)'
                )
                .replace('EDT', 'EST5EDT')
            ),

            str_date=(
                autos.createdOn.str.slice(4, 19)
                + ' ' +
                autos.createdOn.str.slice(-4)
            ),

            createdOn=lambda df_: to_tz(
                df_,
                'str_date',
                'offset',
                'America/New_York'
            ),

            ffs=(
                autos.eng_dscr
                .str.contains('FFS')
            )
        )

        .pipe(get_var, 'new_cols')

        .astype(types)

        .loc[:, final_cols]
    )

In [ ]:
tweak_autos(autos)

print(new_cols)

#### Debugging Chains Part IV - `%debug`

In [ ]:
def err(*args):

    1 / 0

In [ ]:
def tweak_autos(autos):

    cols = [
        'city08', 'comb08', 'highway08', 'cylinders',
        'displ', 'drive', 'eng_dscr', 'fuelCost08',
        'make', 'model', 'trany', 'range',
        'createdOn', 'year'
    ]

    return (
        autos
        [cols]

        .assign(

            cylinders=(
                autos.cylinders
                .fillna(0)
                .astype('int8')
            ),

            displ=(
                autos.displ
                .fillna(0)
                .astype('float16')
            ),

            drive=(
                autos.drive
                .fillna('Other')
                .astype('category')
            ),

            automatic=(
                autos.trany
                .str.contains('Auto')
            ),

            speeds=(
                autos.trany
                .str.extract(r'(?P<speeds>\d+)')
                .fillna('20')
                .astype('int8[pyarrow]')
            ),

            offset=(
                autos.createdOn
                .str.extract(
                    r'\d\d:\d\d (?P<offset>[A-Z]{3}?)'
                )
                .replace('EDT', 'EST5EDT')
            ),

            str_date=(
                autos.createdOn.str.slice(4, 19)
                + ' ' +
                autos.createdOn.str.slice(-4)
            ),

            createdOn=lambda df_: to_tz(
                df_,
                'str_date',
                'offset',
                'America/New_York'
            ),

            ffs=(
                autos.eng_dscr
                .str.contains('FFS')
            )
        )

        .pipe(err)

        .astype({
            'highway08': 'int8',
            'city08': 'int16',
            'comb08': 'int16',
            'fuelCost08': 'int16',
            'range': 'int16',
            'year': 'int16',
            'make': 'category'
        })

        .drop(columns=['trany', 'eng_dscr'])
    )

In [ ]:
res = tweak_autos(autos)

In [ ]:
%debug

#### Debugging Apply (& Friends)

In [ ]:
def err(*args):

    set_trace()

In [ ]:
class DebugException(Exception):

    pass

In [ ]:
def debug_var(
    thing,
    *,
    name='debug_item',
    raise_ex=True
):

    globals()[name] = thing

    if raise_ex:

        raise DebugException

    return thing

In [ ]:
def tweak_autos(autos):

    orig_cols = [
        'city08', 'comb08', 'highway08', 'cylinders',
        'displ', 'drive', 'eng_dscr', 'fuelCost08',
        'make', 'model', 'trany', 'range',
        'createdOn', 'year'
    ]

    types = {
        'highway08': 'int8[pyarrow]',
        'city08': 'int16[pyarrow]',
        'comb08': 'int16[pyarrow]',
        'fuelCost08': 'int16[pyarrow]',
        'range': 'int16[pyarrow]',
        'year': 'int16[pyarrow]',
        'make': 'category',
        'cylinders': 'int8[pyarrow]'
    }

    final_cols = [
        'city08',
        'comb08',
        'highway08',
        'cylinders',
        'displ',
        'drive',
        'fuelCost08',
        'make',
        'model',
        'range',
        'createdOn',
        'year',
        'automatic',
        'speeds',
        'ffs'
    ]

    return (
        autos
        [orig_cols]

        .assign(

            drive=(
                autos.drive
                .fillna('Other')
                .astype('category')
            ),

            automatic=(
                autos.trany
                .str.contains('Auto')
            ),

            speeds=(
                autos.trany
                .str.extract(r'(?P<speeds>\d+)')
                .fillna('20')
                .astype('int8[pyarrow]')
            ),

            offset=(
                autos.createdOn
                .str.extract(
                    r'\d\d:\d\d (?P<offset>[A-Z]{3}?)'
                )
                .replace('EDT', 'EST5EDT')
            ),

            str_date=(
                autos.createdOn.str.slice(4, 19)
                + ' ' +
                autos.createdOn.str.slice(-4)
            ),

            createdOn=lambda df_: to_tz(
                df_,
                'str_date',
                'offset',
                'America/New_York'
            ),

            ffs=(
                autos.eng_dscr
                .str.contains('FFS')
            )
        )

        .astype(types)

        .loc[:, final_cols]
    )

In [ ]:
autos2 = tweak_autos(autos)

In [ ]:
autos2.apply(debug_var, name='this')

In [ ]:
autos2 = tweak_autos(autos)

In [ ]:
autos2.apply(debug_var, name='this')

In [ ]:
this

In [ ]:
autos2.apply(debug_var, axis=1)

In [ ]:
debug_item

In [ ]:
(
    autos2
    .assign(new_col=debug_var)
)

In [ ]:
debug_item

In [ ]:
print(debug_item)

In [ ]:
(
    autos2
    .groupby('make')
    .agg({'city08': debug_var})
)

In [ ]:
debug_item

#### Memory Usage

In [ ]:
dd.info(memory_usage='deep')

In [ ]:
# pip install memory-profiler

In [ ]:
%load_ext memory_profiler

In [ ]:
%%memit

dd = tweak_river(df)

In [ ]:
#### Copy on Write